## Running on multiple GPUs using Hugging Face Transformers

Naive pipeline parallelism is supported out of the box. For this, simply load the model with device="auto" which will automatically place the different layers on the available GPUs.

Your task:

1. Create a pod with two 24GB GPUs.

2. Try to run the model with device="auto" and see how much VRAM is used. You can also try to run the model with device_map="auto" which will automatically place the different layers on the available GPUs. This is a more advanced version of pipeline parallelism that allows for more flexibility in how the model is distributed across GPUs.

In [2]:
!pip install nvidia-ml-py

Looking in indexes: https://mirrors.cernet.edu.cn/pypi/web/simple, https://pypi.ngc.nvidia.com

[notice] A new release of pip is available: 24.2 -> 25.1.1
[notice] To update, run: python -m pip install --upgrade pip


In [ ]:
model_path = "/ssdshare/share/Meta-Llama-3-8B-Instruct/"
# TODO(Your Task): Load the model to multiple GPUs and check the GPU memory usage
import torch
from transformers import AutoModelForCausalLM
import gc, pynvml

def get_gpu_memory_usage():
    """Return GPU memory usage for all available GPUs in GB"""
    pynvml.nvmlInit()
    usage = {}
    count = pynvml.nvmlDeviceGetCount()
    for i in range(count):
        handle = pynvml.nvmlDeviceGetHandleByIndex(i)
        info = pynvml.nvmlDeviceGetMemoryInfo(handle)
        usage[f"gpu_{i}"] = {
            "used": info.used / 1024**3,
            "total": info.total / 1024**3
        }
    pynvml.nvmlShutdown()
    return usage

# Clear caches at the beginning
gc.collect()
torch.cuda.empty_cache()
initial_memory = get_gpu_memory_usage()

# 1. Load model to a single GPU (cuda:0)
print("Loading model to a single GPU (cuda:0)...")
model_single_gpu = AutoModelForCausalLM.from_pretrained(
    model_path,
    device_map="cuda:0",
    torch_dtype=torch.float16
)
single_gpu_memory = get_gpu_memory_usage()
used_diff = single_gpu_memory['gpu_0']['used'] - initial_memory['gpu_0']['used']
print("\nSingle GPU (cuda:0):")
print(f"gpu_0: {used_diff:.2f} GB additional used (from initial {initial_memory['gpu_0']['used']:.2f} GB to {single_gpu_memory['gpu_0']['used']:.2f} GB)")

del model_single_gpu
gc.collect()
torch.cuda.empty_cache()

# 2. Load model with device_map="auto"
# Clear caches before second load and record fresh initial memory
gc.collect()
torch.cuda.empty_cache()
initial_memory_auto = get_gpu_memory_usage()

print("\nLoading model with device_map='auto'...")
model_device_auto = AutoModelForCausalLM.from_pretrained(
    model_path,
    device_map="auto",
    torch_dtype=torch.float16
)
auto_memory = get_gpu_memory_usage()
print("\ndevice_map='auto':")
for gpu, mem in auto_memory.items():
    used_diff = mem['used'] - initial_memory_auto[gpu]['used']
    print(f"{gpu}: {used_diff:.2f} GB additional used (from initial {initial_memory_auto[gpu]['used']:.2f} GB to {mem['used']:.2f} GB)")

# The task is mainly completed by Claude 3.7 Sonnet Thinking, by prompt "complete the task. compare the VRAM usage before and after loading.".

Loading model to a single GPU (cuda:0)...


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]


Single GPU (cuda:0):
gpu_0: 17.36 GB additional used (from initial 0.44 GB to 17.80 GB)

Loading model with device_map='auto'...


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]


device_map='auto':
gpu_0: 6.67 GB additional used (from initial 0.89 GB to 7.56 GB)
gpu_1: 8.68 GB additional used (from initial 0.45 GB to 9.12 GB)


The GPU memory usage of loading the model to only one GPU is 17.36GB.

The GPU memory usage of loading the model with device="auto" is \_\_\_\_\_\_\_. The GPU memory usage of loading the model with device_map="auto" is 6.67GB+8.68GB=15.35GB.

The number of GPUs you used is 2.

Does the numbers above make sense?